# Serveur voix baoulé — RunPod (ADR-0045)

**But** : le paysan **parle** en baoulé sur WhatsApp et **entend** la réponse **avec la voix du locuteur natif**.
- `POST /transcribe` : vocal → texte baoulé, par le 1B fine-tuné (fiche 0007, CER 13,4 %), avec le modèle de langue s'il gagne sur dev (étape 3 bis) ;
- `GET /audio/<nom>.ogg` : les réponses **lues par le locuteur** pendant la dictée (aucune voix de synthèse).

**Pod RunPod** : 1× **A40** (ou toute carte ≥ 16 Go **disponible**), template **Runpod Pytorch 2.8.0**, **disque de 40 GB**
(*Container disk*, ou *Volume disk* sur `/workspace`), **On-Demand**. ⚠️ Dans *Edit template* → **Expose HTTP Ports**, ajouter **`8000`** (à côté de `8888`).

**Avant de lancer** : téléverser `dictee_bci_433.zip` (export de l'écran Dictée) dans `/workspace`, et avoir un jeton Hugging Face
**en lecture seule** (le modèle est dans un dépôt privé). Le jeton est demandé par une saisie masquée, jamais écrit.

**Étape 3 bis (facultative, ~25 min)** : mesure du modèle de langue. Il faut ta **clé API Mozilla Data Collective** (Common Voice). Sans elle, saute la cellule 7b : le serveur décode sans modèle de langue.

Exécuter les cellules **dans l'ordre**, une à la fois. Le pod est facturé tant qu'il existe : **Terminate** après la démo.

In [ ]:
# Cellule 1 — chemins, réglages et lanceur commun (re-exécuter si le noyau redémarre)
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/workspace") if Path("/workspace").is_dir() else Path.cwd()
os.chdir(WORK)
VENV = WORK / "omni312"                   # Python 3.12 dédié (fairseq2 0.6 n'existe pas en 3.13)
VENV_PY = str(VENV / "bin" / "python")
CKPT_ROOT = WORK / "wourri_ft"            # modèle fine-tuné (~3,9 Go)
ASSETS = WORK / "fairseq2_assets"         # carte fairseq2 du modèle
AUDIO_DIR = WORK / "reponses_ogg"         # réponses lues par le locuteur, en ogg/opus
EXPORT_ZIP = WORK / "dictee_bci_433.zip"  # export de l'écran Dictée (à téléverser)
HF_REPO = "seven990/omniasr-ctc-1b-bci-wourri-cv27"   # dépôt Hugging Face PRIVÉ (fiche 0007, 2e fine-tune)
STEP = "step_3000"                        # version retenue sur dev (fiche 0007 : CER 13,4 % contre 16,6 %)
PORT = 8000
LM_DIR = WORK / "lm"                       # modèle de langue retenu (cellule 7b)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # hf_transfer absent du Python 3.12

def clean_env(extra=None):
    """Environnement des sous-processus SANS jetons."""
    env = {k: v for k, v in os.environ.items()
           if not any(s in k.upper() for s in ("TOKEN", "SECRET", "PASSWORD", "CREDENTIAL", "API_KEY"))}
    env["PYTHONUNBUFFERED"] = "1"
    env.update(extra or {})
    return env

def sh(args, cwd=None, env=None, check=True):
    """Lance une commande en affichant sa sortie en direct ; retourne le code."""
    proc = subprocess.Popen([str(a) for a in args], cwd=cwd, env=env or clean_env(), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if check and code != 0:
        raise SystemExit(f"ECHEC (code {code}) : {' '.join(str(a) for a in args)[:200]}")
    return code

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "PAS DE GPU")
print("Pod :", os.environ.get("RUNPOD_POD_ID", "(inconnu)"))
print(">>> CELLULE 1 OK -> cellule 2.")

## Étape 1 — Installation (~5 à 7 min)

ffmpeg (décodage des vocaux WhatsApp), Python 3.12 et les versions figées du benchmark 0003, plus le serveur web.

In [ ]:
# Cellule 2 — ffmpeg + Python 3.12 (uv) + omnilingual-asr 0.1.0 + serveur web (FastAPI)
sh(["bash", "-c", "command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -qq ffmpeg >/dev/null)"])
UV = [sys.executable, "-m", "uv"]
# huggingface_hub dans le Python du noyau : la cellule 7 y télécharge le modèle (absent du template RunPod).
sh([sys.executable, "-m", "pip", "install", "--quiet", "uv", "huggingface_hub"])
if not VENV.exists():
    sh(UV + ["venv", "--quiet", "--python", "3.12", VENV])
sh(UV + ["pip", "install", "--quiet", "--no-cache", "--python", VENV_PY,
         "omnilingual-asr==0.1.0", "fairseq2[arrow]==0.6", "torch==2.8.0", "torchaudio==2.8.0",
         "datasets<4.0", "soundfile", "librosa<1.0", "jiwer", "pyarrow==20.0.0",
         "fastapi", "uvicorn", "python-multipart", "requests", "pyctcdecode==0.5.0", "sentencepiece"])
sh([VENV_PY, "-c", "import sys, torch, fastapi, omnilingual_asr; print('Python', sys.version.split()[0], "
    "'| torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN')"])
sh(["bash", "-c", "ffmpeg -version | head -1"], check=False)
print("\n>>> INSTALLATION TERMINÉE -> cellule 3.")

## Étape 2 — Les modules du dépôt

Les cellules 3 à 6 écrivent sept fichiers du dépôt (`finetune/`), copies identiques vérifiées par un test. Il suffit de les exécuter.

In [ ]:
%%writefile dictee_to_parquet.py
"""Convertit l'export dictée (ZIP audiofolder) en dataset parquet Omnilingual — maillon ③ (#474).

Pont entre le maillon ② (export dictée, ADR-0035) et le maillon ③ (fine-tune
Omnilingual, cf. étude de faisabilité docs/benchmarks/0004 et test mécanique 0005).

Entrée : ZIP export dictée
    audio/<x>.webm ...
    metadata.csv : file_name, transcription, language, filiere, text_fr

Sortie : un dataset parquet partitionné (« hive »), seul format que lit le chargeur
`MixtureParquetStorage` de la recette de fine-tune (omnilingual-asr 0.1.0) :
    <racine>/corpus=<corpus>/split=<split>/language=<code>/part-0.parquet
        - text        : transcription normalisée (baoulé) — la CIBLE ASR
        - audio_bytes : audio FLAC 16 kHz mono, en list<int8> (schéma amont)
        - audio_size  : nombre d'échantillons du waveform décodé
    corpus / split / language vivent dans les NOMS DE DOSSIERS, pas dans le fichier :
    la recette découvre les splits et les partitions à partir de l'arborescence.
    Un fichier parquet « à plat » n'est pas lu (aucun split découvert).

Le code langue est celui d'Omnilingual (ISO 639-3 + écriture) : `bci` -> `bci_Latn`.

Usage :
    python dictee_to_parquet.py export.zip dataset_root \\
        --corpus wourri_dictee --split train --language bci \\
        --stats language_distribution_0.tsv

Décodage : soundfile (wav/flac/ogg) ; repli librosa+ffmpeg pour webm/opus/mp3
(le fine-tune tourne sur Colab/Kaggle où ffmpeg est présent). Le rééchantillonnage
utilise librosa si présent, sinon un repli linéaire (dégradé — le vrai run a librosa).

Ce module est recopié tel quel dans le notebook `colab/omnilingual_finetune_smoke_test.ipynb`
(cellule %%writefile) : un test vérifie que les deux copies restent identiques.
"""
from __future__ import annotations

import argparse
import csv
import io
import re
import sys
import unicodedata
import zipfile
from pathlib import Path, PurePosixPath

TARGET_SR = 16000
ROW_GROUP_SIZE = 100  # valeur amont : limite la mémoire du streaming et permet le mélange
FILE_COLUMNS = ["text", "audio_bytes", "audio_size"]

# Apostrophe baoulé : Common Voice bci et la sortie d'Omnilingual l'écrivent U+02BC (ʼ,
# lettre modificative, conservée par la normalisation amont et présente dans le vocabulaire :
# le 1B la produit au benchmark 0003). Les variantes tapées au clavier ENTRE DEUX LETTRES
# (' ’ ‘ ‛) y sont ramenées, sinon un même son aurait plusieurs cibles ; ailleurs ce sont
# des guillemets, retirés comme la ponctuation. Un ton combinant (ɛ̀ = ɛ + U+0300, sans forme
# précomposée) compte comme lettre.
_APOSTROPHES = "'‘’‛"
_INNER_APOSTROPHE = re.compile(rf"(?<=[\ẁ-ͯ])[{_APOSTROPHES}](?=\w)")
_OTHER_APOSTROPHE = re.compile(rf"[{_APOSTROPHES}]")
# Invisibles supprimés par l'amont (shared_deletion_list) + BOM : souvent hérités d'un copier-coller.
_INVISIBLE = re.compile("[​‌‎‏‪‬﻿]")
_PUNCT = re.compile(r"[.,!?;:«»\"“”„()\[\]{}…—–/\\_]")
_DIGIT_WORD = re.compile(r"(?<!\S)\d+(?!\S)")


def normalize_text(text: str) -> str:
    """Normalise une transcription comme la préparation de données amont (text_tools) :
    NFKC, invisibles supprimés, minuscules, ponctuation retirée, mots uniquement numériques
    retirés. Les lettres baoulé (ɛ ɔ ɲ, tons, ʼ) sont conservées : elles portent le sens."""
    t = unicodedata.normalize("NFKC", text or "")
    t = _INVISIBLE.sub("", t).lower()
    t = _INNER_APOSTROPHE.sub("ʼ", t)
    t = _OTHER_APOSTROPHE.sub(" ", t)
    t = _PUNCT.sub(" ", t)
    t = _DIGIT_WORD.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()


def omni_language_code(code: str) -> str:
    """Code Omnilingual (`lang_ids.py`) = ISO 639-3 + écriture : 'bci' -> 'bci_Latn'.

    Les langues de l'atelier s'écrivent en latin (colonne `script` de la table
    `languages`) ; un code déjà complet ('bci_Latn') est gardé tel quel."""
    code = (code or "").strip()
    if not code:
        raise ValueError("code langue vide")
    return code if "_" in code else f"{code}_Latn"


def decode_to_16k_mono(raw: bytes, filename: str):
    """Octets audio -> (waveform float32 mono 16 kHz, n_samples)."""
    import numpy as np
    import soundfile as sf

    try:
        wav, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)  # (n, ch)
        wav = wav.mean(axis=1)  # -> mono (n,)
    except Exception:
        wav, sr = _decode_via_librosa(raw, filename)  # webm/opus/mp3 -> ffmpeg
    if sr != TARGET_SR:
        wav = _resample(np.asarray(wav, dtype="float32"), sr, TARGET_SR)
    return np.asarray(wav, dtype="float32"), int(len(wav))


def _decode_via_librosa(raw: bytes, filename: str):
    import tempfile
    import warnings

    import librosa

    suffix = Path(filename).suffix or ".webm"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(raw)
        path = tmp.name
    try:
        # webm : soundfile échoue toujours, librosa passe par audioread + ffmpeg. Ses deux
        # avertissements (« PySoundFile failed », dépréciation d'audioread) sortiraient une fois
        # par clip et noieraient la sortie du notebook (433 clips = 433 paires).
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            warnings.simplefilter("ignore", FutureWarning)
            wav, sr = librosa.load(path, sr=None, mono=True)  # float32 mono
    finally:
        Path(path).unlink(missing_ok=True)
    return wav, sr


def _resample(wav, sr: int, target: int):
    import numpy as np

    try:
        import librosa

        return librosa.resample(wav, orig_sr=sr, target_sr=target)
    except Exception:
        # repli linéaire (le vrai run a librosa ; ce repli garde le script utilisable en test)
        n = int(round(len(wav) * target / sr))
        if n <= 0:
            return wav
        return np.interp(
            np.linspace(0, len(wav) - 1, n), np.arange(len(wav)), wav
        ).astype("float32")


def encode_flac(wav) -> bytes:
    """Waveform float32 mono 16 kHz -> octets FLAC."""
    import soundfile as sf

    buf = io.BytesIO()
    sf.write(buf, wav, TARGET_SR, format="FLAC")
    return buf.getvalue()


def read_export(zip_path) -> list[dict]:
    """Lit le ZIP export dictée -> [{file_name, transcription, language, filiere, text_fr, raw}]."""
    rows: list[dict] = []
    with zipfile.ZipFile(zip_path) as zf:
        names = set(zf.namelist())
        metas = sorted(n for n in names if PurePosixPath(n).name == "metadata.csv")
        if not metas:
            raise SystemExit("metadata.csv introuvable dans le ZIP")
        meta_name = "metadata.csv" if "metadata.csv" in names else metas[0]
        meta = zf.read(meta_name).decode("utf-8-sig")
        for r in csv.DictReader(io.StringIO(meta)):
            fn = (r.get("file_name") or "").strip()
            if not fn:
                continue
            if fn not in names:
                # tolère un préfixe de chemin différent, jamais un autre fichier :
                # même nom exact (pas « 11.webm » pour « 1.webm ») et un seul candidat
                base = PurePosixPath(fn).name
                cands = [n for n in names if PurePosixPath(n).name == base]
                if len(cands) != 1:
                    why = "absent du ZIP" if not cands else f"ambigu ({len(cands)} fichiers)"
                    print(f"  [SKIP] audio {why} : {fn}", file=sys.stderr)
                    continue
                fn = cands[0]
            rows.append(
                {
                    "file_name": fn,
                    "transcription": (r.get("transcription") or "").strip(),
                    "language": (r.get("language") or "").strip(),
                    "filiere": (r.get("filiere") or "").strip(),
                    "text_fr": (r.get("text_fr") or "").strip(),
                    "raw": zf.read(fn),
                }
            )
    return rows


def make_table(examples):
    """[(transcription, waveform float32 mono 16 kHz)] -> table parquet (colonnes FILE_COLUMNS).

    Ignore les paires sans texte après normalisation ou sans audio : une paire
    audio↔texte complète est requise pour l'apprentissage."""
    import numpy as np
    import pyarrow as pa

    texts, audios, sizes = [], [], []
    for text, wav in examples:
        norm = normalize_text(text)
        if not norm or len(wav) == 0:
            continue
        texts.append(norm)
        audios.append(np.frombuffer(encode_flac(wav), dtype=np.int8))
        sizes.append(int(len(wav)))
    return pa.table(
        {
            "text": pa.array(texts, type=pa.string()),
            "audio_bytes": pa.array(audios, type=pa.list_(pa.int8())),
            "audio_size": pa.array(sizes, type=pa.int64()),
        }
    )


def write_partition(table, root, corpus: str, split: str, language: str) -> Path:
    """Écrit la table dans <root>/corpus=…/split=…/language=…/part-0.parquet.

    Remplace les parquet déjà présents dans CETTE partition : un export dictée est
    toujours complet, le reconvertir doit remplacer l'ancien, pas s'y ajouter."""
    import pyarrow.parquet as pq

    for key, value in (("corpus", corpus), ("split", split), ("language", language)):
        if not value or "/" in value or "=" in value:
            raise ValueError(f"{key} invalide pour un nom de partition : {value!r}")
    if "_" in split:
        # la recette lit un split « <split>_<corpus> » comme un filtre de corpus
        raise ValueError(f"split sans '_' attendu (reçu {split!r})")
    if table.num_rows == 0:
        # une partition vide fait échouer la recette au démarrage (lecture de son 1er exemple)
        raise ValueError(f"aucune paire audio↔texte exploitable pour {corpus}/{split}")
    part_dir = Path(root) / f"corpus={corpus}" / f"split={split}" / f"language={language}"
    part_dir.mkdir(parents=True, exist_ok=True)
    for old in part_dir.glob("*.parquet"):
        old.unlink()
    out = part_dir / "part-0.parquet"
    pq.write_table(table, out, row_group_size=ROW_GROUP_SIZE)
    return out


def write_stats(root, out_tsv) -> list[dict]:
    """Écrit le TSV `corpus / language / hours` attendu par `dataset_summary_path`
    (pondération des partitions à l'entraînement). Comme l'outil amont, les heures
    sont cumulées sur tous les splits."""
    import pyarrow.dataset as ds

    table = ds.dataset(str(root), format="parquet", partitioning="hive").to_table(
        columns=["corpus", "language", "audio_size"]
    )
    samples: dict[tuple[str, str], int] = {}
    for corpus, language, size in zip(
        table.column("corpus").to_pylist(),
        table.column("language").to_pylist(),
        table.column("audio_size").to_pylist(),
    ):
        key = (str(corpus), str(language))
        samples[key] = samples.get(key, 0) + int(size)
    stats = [
        {"corpus": c, "language": lang, "hours": n / TARGET_SR / 3600}
        for (c, lang), n in sorted(samples.items())
    ]
    lines = ["corpus\tlanguage\thours"]
    lines += [f"{s['corpus']}\t{s['language']}\t{s['hours']:.6f}" for s in stats]
    Path(out_tsv).parent.mkdir(parents=True, exist_ok=True)
    Path(out_tsv).write_text("\n".join(lines) + "\n", encoding="utf-8")
    return stats


def build_dataset(rows, root, corpus: str, split: str, language: str) -> int:
    """Lignes d'export dictée -> partition parquet. Retourne le nombre de clips écrits."""

    def examples():
        for r in rows:
            if r.get("transcription"):
                wav, _ = decode_to_16k_mono(r["raw"], r["file_name"])
                yield r["transcription"], wav

    table = make_table(examples())
    write_partition(table, root, corpus, split, omni_language_code(language))
    return table.num_rows


def main():
    ap = argparse.ArgumentParser(
        description="Export dictée (ZIP audiofolder) -> dataset parquet Omnilingual (#474)"
    )
    ap.add_argument("zip_path", help="ZIP export dictée (audio/ + metadata.csv)")
    ap.add_argument("out_root", help="racine du dataset (le `data:` de la carte fairseq2)")
    ap.add_argument("--corpus", default="wourri_dictee")
    ap.add_argument("--split", default="train", choices=["train", "dev", "test"])
    ap.add_argument("--language", default=None, help="code langue (ex. bci) ; défaut : celui du CSV")
    ap.add_argument("--stats", default=None, help="écrit aussi le TSV corpus/language/hours")
    args = ap.parse_args()

    rows = read_export(args.zip_path)
    language = args.language
    if not language:
        found = {r["language"] for r in rows if r.get("language")}
        if len(found) != 1:
            raise SystemExit(f"--language requis : langues trouvées dans le CSV = {sorted(found)}")
        language = found.pop()
    n = build_dataset(rows, args.out_root, args.corpus, args.split, language)
    print(f"OK : {n} clips -> {args.out_root} (corpus={args.corpus}, split={args.split}, "
          f"language={omni_language_code(language)})")
    if args.stats:
        write_stats(args.out_root, args.stats)
        print(f"Stats -> {args.stats}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile prepare_bci_dataset.py
"""Prépare le dataset baoulé du fine-tune Omnilingual (#474, ADR-0044).

Common Voice bci (miroir Klayt, révision épinglée du benchmark 0003) :
    train -> split « train », validation -> « dev », test -> « test » (avec --with-test)
+ l'export de l'écran Dictée de l'atelier (optionnel) -> corpus « wourri_dictee », split « train »,
puis le TSV de statistiques et la carte dataset fairseq2.

Le split « test » (290 clips, ceux du benchmark 0003) n'est JAMAIS entraîné : la recette
n'entraîne que sur `train_split: train`. Il est seulement évalué (`valid_split: dev,test`) pour
comparer, dans les mêmes conditions, le modèle de départ et le modèle fine-tuné. La séparation
locuteurs / phrases entre ces splits est vérifiée (docs/benchmarks/0005 §3).

Usage (dans le Python 3.12 du notebook) :
    python prepare_bci_dataset.py --work /workspace --dictee /workspace --with-test

Ce module est recopié tel quel dans les notebooks de finetune/colab/ (cellule %%writefile) :
`python finetune/colab/sync_smoke_test_notebook.py` les resynchronise, un test le vérifie.
"""
from __future__ import annotations

import argparse
import csv
import io
import json
import shutil
import tarfile
import zipfile
from pathlib import Path

import dictee_to_parquet as d2p

DATASET_REPO = "Klayt/baoule-common-voice"
REV = "176d5f8ad6c04e5ca1a0e66770866709bcbb338b"  # même révision que le benchmark 0003
# split Hugging Face -> split de la recette (« dev » : nom attendu par valid_split)
CV_SPLITS = {"train": "train", "validation": "dev", "test": "test"}
CARD_NAME = "wourri_bci"


def get_audio(row):
    """Audio d'une ligne Hugging Face -> (waveform float32 mono, sample rate)."""
    import numpy as np

    a = row["audio"]
    if isinstance(a, dict):  # datasets < 4.0
        return np.asarray(a["array"], dtype="float32"), int(a["sampling_rate"])
    samples = a.get_all_samples()  # datasets >= 4.0 (AudioDecoder)
    return samples.data.numpy().squeeze().astype("float32"), int(samples.sample_rate)


def cv_examples(hf_split, load_dataset=None):
    """[(phrase, waveform 16 kHz)] du split Common Voice demandé."""
    if load_dataset is None:
        from datasets import load_dataset
    for row in load_dataset(DATASET_REPO, revision=REV, split=hf_split):
        wav, sr = get_audio(row)
        if sr != d2p.TARGET_SR:
            wav = d2p._resample(wav, sr, d2p.TARGET_SR)
        yield row["sentence"], wav


def find_export(path, tmp_dir) -> Path:
    """Le ZIP de l'écran Dictée, ou son dossier décompressé (rezippé au format attendu)."""
    src = Path(path)
    if src.is_file():
        return src
    zips = sorted(src.rglob("dictee_*.zip"))
    if zips:
        return zips[0]
    metas = sorted(src.rglob("metadata.csv"))
    if metas:
        base = Path(tmp_dir) / "dictee_export"
        return Path(shutil.make_archive(str(base), "zip", root_dir=metas[0].parent))
    raise SystemExit(f"Ni dictee_*.zip ni metadata.csv sous {path}")


def write_card(asset_dir, data_root) -> Path:
    """Carte fairseq2 `mixture_parquet_asr_dataset` (répertoire FAIRSEQ2_USER_ASSET_DIR)."""
    asset_dir = Path(asset_dir)
    asset_dir.mkdir(parents=True, exist_ok=True)
    card = asset_dir / f"{CARD_NAME}.yaml"
    card.write_text(
        f"name: {CARD_NAME}\n"
        "dataset_family: mixture_parquet_asr_dataset\n"
        "dataset_config:\n"
        f"  data: {data_root}\n"
        "tokenizer_ref: omniASR_tokenizer\n",
        encoding="utf-8",
    )
    return card


def prepare(work, dictee=None, with_test=False, load_dataset=None) -> dict:
    """Construit <work>/wourri_bci (dataset partitionné + TSV) et la carte fairseq2.

    Repart toujours d'un dossier vide : une préparation précédente (autres splits,
    ancien export) ne doit pas se mélanger à celle-ci. Retourne les comptes par partition.
    """
    work = Path(work)
    dataset_dir = work / "wourri_bci"
    data_root = dataset_dir / "version=0"  # le `data:` de la carte fairseq2
    shutil.rmtree(dataset_dir, ignore_errors=True)
    lang = d2p.omni_language_code("bci")
    counts = {}

    for hf_split, split in CV_SPLITS.items():
        if split == "test" and not with_test:
            continue
        table = d2p.make_table(cv_examples(hf_split, load_dataset))
        d2p.write_partition(table, data_root, "cv_bci", split, lang)
        counts[f"cv_bci/{split}"] = table.num_rows

    if dictee:
        export = find_export(dictee, work)
        rows = d2p.read_export(export)
        counts["wourri_dictee/train"] = d2p.build_dataset(rows, data_root, "wourri_dictee", "train", "bci")

    stats = d2p.write_stats(data_root, dataset_dir / "language_distribution_0.tsv")
    card = write_card(work / "fairseq2_assets", data_root)
    return {"counts": counts, "stats": stats, "card": card}


ZENODO_DIR = "bci-datasets"  # racine de l'archive Zenodo 6705861 (data354, CC BY 4.0)


def zenodo_rows(zip_path) -> list[tuple[str, str]]:
    """[(fichier audio, transcription)] de l'archive Zenodo. Les numéros de verset en tête
    (« 28 Amun… ») ne sont pas prononcés (mesuré le 2026-10-24 sur les durées : +0,02 s à
    longueur égale) : `normalize_text` retire les mots numériques, rien d'autre à faire."""
    with zipfile.ZipFile(zip_path) as z:
        tsv = z.read(f"{ZENODO_DIR}/transcription.tsv").decode("utf-8-sig")
    return [(r["id"], r["bci"]) for r in csv.DictReader(io.StringIO(tsv), delimiter="\t") if r.get("bci")]


def zenodo_examples(zip_path, rows):
    with zipfile.ZipFile(zip_path) as z:
        for fichier, texte in rows:
            wav, _ = d2p.decode_to_16k_mono(z.read(f"{ZENODO_DIR}/wav/{fichier}"), fichier)
            yield texte, wav


def cv27_examples(rows, dossier):
    for r in rows:
        raw = (Path(dossier) / "clips" / r["path"]).read_bytes()
        wav, _ = d2p.decode_to_16k_mono(raw, r["path"])
        yield r["sentence"], wav


def extraire_cv27(source, work) -> Path:
    """Dossier `…/bci` de Common Voice 27 : l'archive .tar.gz est extraite une fois dans <work>/cv27."""
    import cv27_split as cvs  # import local : les notebooks du 1er fine-tune n'embarquent pas ce module

    source = Path(source)
    if source.is_file():
        cible = Path(work) / "cv27"
        if not any(cible.rglob("validated.tsv")):
            with tarfile.open(source) as t:
                t.extractall(cible, filter="data")
        source = cible
    return cvs.trouver(source)


def prepare_cv27(work, cv27, zenodo=None, dictee=None) -> dict:
    """2e fine-tune : Common Voice 27 réparti par `cv27_split` (train / dev / test), + Zenodo et
    dictée en train. La répartition est vérifiée contre TOUS les textes d'entraînement
    (aucune phrase du test chez Zenodo ni dans la dictée) avant d'écrire quoi que ce soit."""
    import cv27_split as cvs  # import local : les notebooks du 1er fine-tune n'embarquent pas ce module

    work = Path(work)
    dataset_dir = work / "wourri_bci"
    data_root = dataset_dir / "version=0"
    shutil.rmtree(dataset_dir, ignore_errors=True)
    lang = d2p.omni_language_code("bci")
    zen = zenodo_rows(zenodo) if zenodo else []
    dic = d2p.read_export(find_export(dictee, work)) if dictee else []
    dossier = extraire_cv27(cv27, work)
    rep, manifeste = cvs.preparer(dossier, [t for _, t in zen] + [r["transcription"] for r in dic])
    counts = {}
    for split, rows in rep.items():
        table = d2p.make_table(cv27_examples(rows, dossier))
        d2p.write_partition(table, data_root, "cv_bci", split, lang)
        counts[f"cv_bci/{split}"] = table.num_rows
    if zen:
        table = d2p.make_table(zenodo_examples(zenodo, zen))
        d2p.write_partition(table, data_root, "zenodo_bci", "train", lang)
        counts["zenodo_bci/train"] = table.num_rows
    if dic:
        counts["wourri_dictee/train"] = d2p.build_dataset(dic, data_root, "wourri_dictee", "train", "bci")
    (work / "split_cv27.json").write_text(json.dumps(manifeste, ensure_ascii=False, indent=1), encoding="utf-8")
    stats = d2p.write_stats(data_root, dataset_dir / "language_distribution_0.tsv")
    card = write_card(work / "fairseq2_assets", data_root)
    return {"counts": counts, "stats": stats, "card": card, "splits": manifeste["splits"]}


def main():
    ap = argparse.ArgumentParser(description="Dataset baoulé du fine-tune Omnilingual (#474)")
    ap.add_argument("--work", required=True, help="dossier de travail (sortie)")
    ap.add_argument("--dictee", default=None, help="ZIP de la dictée ou dossier qui le contient")
    ap.add_argument("--with-test", action="store_true",
                    help="ajoute le split test (évalué, jamais entraîné)")
    ap.add_argument("--cv27", default=None, help="Common Voice 27 bci (.tar.gz ou dossier extrait) : 2e fine-tune")
    ap.add_argument("--zenodo", default=None, help="archive Zenodo 6705861 (bci-datasets.zip), avec --cv27")
    args = ap.parse_args()
    if args.cv27:
        result = prepare_cv27(args.work, args.cv27, args.zenodo, args.dictee)
        print("Répartition :", json.dumps(result["splits"], ensure_ascii=False))
    else:
        result = prepare(args.work, args.dictee, args.with_test)
    for name, n in result["counts"].items():
        print(f"{name}: {n} clips")
    for s in result["stats"]:
        print(f"{s['corpus']:15s} {s['language']}  {s['hours']:.2f} h")
    print("Carte :", result["card"])


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval_bci_cer.py
"""CER/WER baoulé d'un modèle Omnilingual, mesuré À L'IDENTIQUE du benchmark 0003 (#474, BAO-2).

Critère de BAO-2 (docs/wouri-planning/10) : le CER du modèle fine-tuné, mesuré sur le même jeu
(Common Voice bci test, 290 clips) et comparé aux 22,1 % du 1B de départ. L'UER de la recette
d'entraînement (fiche 0006) n'utilise pas la même normalisation : ce module refait la mesure avec
le protocole du benchmark 0003 :
- mêmes clips : révision épinglée, split test complet, aucun clip exclu ;
- même normalisation et même score (`normalize`, `score` : copie exacte de la cellule 3 de
  finetune/colab/asr_baoule_benchmark.ipynb, vérifiée par un test) ;
- même pipeline : ASRInferencePipeline, un clip à la fois, langue bci_Latn, dtype par défaut.

Le 1B de départ est remesuré dans la même session (témoin) : s'il ne retrouve pas ~22,1 %,
l'environnement diffère de celui du benchmark et la comparaison n'est pas recevable.

Le checkpoint fine-tuné est chargé par une carte fairseq2 qui hérite d'omniASR_CTC_1B (famille,
architecture, tokenizer) et pointe vers le dossier `step_N/model` écrit par l'entraînement
(pp_00/tp_00/sdp_00.pt, lu par le chargeur natif de fairseq2 0.6).

Usage (dans le Python 3.12 du notebook, après téléchargement du checkpoint) :
    python eval_bci_cer.py --checkpoint-root /tmp/wourri_ft --step step_1200

Ce module est recopié tel quel dans le notebook d'évaluation (cellule %%writefile) :
`python finetune/colab/sync_smoke_test_notebook.py` le resynchronise, un test le vérifie.
"""
from __future__ import annotations

import argparse
import gc
import json
import os
import re
import tempfile
import time
import unicodedata
from pathlib import Path

import jiwer

from prepare_bci_dataset import DATASET_REPO, REV, get_audio

BASE_CARD = "omniASR_CTC_1B"
# CER du 1B de départ au benchmark 0003 (docs/benchmarks/0003-asr-baoule-evaluation-results.md §1)
BENCHMARK_0003_CER = 0.221
BENCHMARK_0003_N = 290
# Écart toléré pour le témoin : au-delà, l'environnement n'est pas celui du benchmark 0003
CONTROL_TOLERANCE = 0.005
N_SAMPLES = 10

# ---- Copie exacte du benchmark 0003 (cellule 3) : ne pas modifier, un test compare les deux ----
# --- normalisation TOLERANTE : minuscule + ponctuation ASCII retiree.
# On CONSERVE les caracteres baoule (ɛ ɔ ʼ, tons) : ils portent le sens. ---
_PUNCT = re.compile(r"[.,!?;:«»\"'`()\[\]{}…—–/\\]")
def normalize(s):
    s = unicodedata.normalize("NFC", (s or "")).lower().strip()
    s = _PUNCT.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

def score(refs, hyps):
    R, H = [], []
    for r, h in zip(refs, hyps):
        rn = normalize(r)
        if rn:                       # jiwer plante sur reference vide
            R.append(rn); H.append(normalize(h))
    return {"cer": float(jiwer.cer(R, H)), "wer": float(jiwer.wer(R, H)), "n": len(R)}
# ---- fin de la copie ----


def find_checkpoint(root, step) -> Path:
    """Dossier `<step>/model` du checkpoint téléchargé (contient pp_00/tp_00/sdp_00.pt)."""
    hits = sorted({p.parent.parent.parent for p in Path(root).rglob("sdp_*.pt")
                   if p.parent.parent.parent.parent.name == step})
    if len(hits) != 1:
        raise SystemExit(f"{len(hits)} checkpoint(s) {step}/model trouvé(s) sous {root} : 1 attendu")
    return hits[0].resolve()


def write_card(asset_dir, step, model_dir, name=None) -> str:
    """Carte fairseq2 du modèle fine-tuné : hérite de BASE_CARD, remplace son checkpoint.
    `name` : obligatoire quand deux checkpoints portent la même étape (2e fine-tune)."""
    if not (Path(model_dir) / "pp_00").is_dir():
        raise SystemExit(f"{model_dir} n'est pas un checkpoint fairseq2 (pp_00/ absent)")
    name = name or f"wourri_bci_ctc_1b_{step}"
    asset_dir = Path(asset_dir)
    asset_dir.mkdir(parents=True, exist_ok=True)
    (asset_dir / f"{name}.yaml").write_text(
        f"name: {name}\nbase: {BASE_CARD}\ncheckpoint: {Path(model_dir).resolve().as_posix()}\n",
        encoding="utf-8",
    )
    return name


def load_test_clips(load_dataset=None, limit=None):
    """(phrases de référence, [(waveform, sample rate)]) du split test, dans l'ordre du dataset."""
    if load_dataset is None:
        from datasets import load_dataset
    rows = list(load_dataset(DATASET_REPO, revision=REV, split="test"))
    rows = rows[:limit] if limit else rows
    return [r["sentence"] for r in rows], [get_audio(r) for r in rows]


def load_cv27_test(manifeste, cv27_dir, limit=None):
    """Test du 2e fine-tune (finetune/cv27_split.py) : mêmes sorties que `load_test_clips`."""
    import dictee_to_parquet as d2p

    test = json.loads(Path(manifeste).read_text(encoding="utf-8"))["test"]
    test = test[:limit] if limit else test
    clips = Path(cv27_dir) / "clips"
    audios = [(d2p.decode_to_16k_mono((clips / t["path"]).read_bytes(), t["path"])[0], d2p.TARGET_SR) for t in test]
    return [t["sentence"] for t in test], audios


def _default_pipeline(card):
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

    return ASRInferencePipeline(model_card=card)


def _free_gpu():
    gc.collect()
    try:
        import torch
    except ImportError:
        return
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def transcribe(card, paths, make_pipeline=None):
    """Transcriptions d'un modèle comme au benchmark 0003 (un clip à la fois, bci_Latn) + durées."""
    make_pipeline = make_pipeline or _default_pipeline
    t0 = time.time()
    pipe = make_pipeline(card)
    load_s = time.time() - t0
    t1 = time.time()
    hyps = list(pipe.transcribe(paths, lang=["bci_Latn"] * len(paths), batch_size=1))
    infer_s = time.time() - t1
    del pipe  # libère le modèle avant de charger le suivant (deux 1B ne tiennent pas sur un T4)
    _free_gpu()
    return hyps, round(load_s, 1), round(infer_s, 1)


def evaluate(models, out_path, load_dataset=None, make_pipeline=None, limit=None, clips=None):
    """Mesure chaque (libellé, carte) sur les mêmes clips ; résultats écrits après chaque modèle.
    `clips` : (références, audios) déjà chargés (test du 2e fine-tune), sinon le test 0003."""
    import soundfile as sf

    refs, audios = clips if clips is not None else load_test_clips(load_dataset, limit)
    tmp = tempfile.mkdtemp()
    paths = []
    for i, (wav, sr) in enumerate(audios):
        p = os.path.join(tmp, f"{i}.wav")
        sf.write(p, wav, sr)
        paths.append(p)
    out_path = Path(out_path)
    results = {}
    for label, card in models:
        hyps, load_s, infer_s = transcribe(card, paths, make_pipeline)
        r = score(refs, hyps)
        r.update(card=card, load_s=load_s, infer_s=infer_s,
                 empty_hyps=sum(1 for h in hyps if not normalize(h)),
                 samples=[{"ref": refs[i], "hyp": hyps[i]} for i in range(min(N_SAMPLES, len(hyps)))],
                 hyps=hyps)
        results[label] = r
        out_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"{label}: CER={r['cer'] * 100:.1f}%  WER={r['wer'] * 100:.1f}%  n={r['n']}  "
              f"vides={r['empty_hyps']}  (charg {load_s:.0f}s, infer {infer_s:.0f}s)", flush=True)
    return results


def control_ok(cer, n):
    """Le témoin (1B de départ) reproduit-il le benchmark 0003 : mêmes clips, ~22,1 % ?"""
    return n == BENCHMARK_0003_N and abs(cer - BENCHMARK_0003_CER) <= CONTROL_TOLERANCE


def report(results, base_label, ft_label, reference_0003=True):
    """Tableau, verdict du témoin, gain et exemples côte à côte (texte à recopier dans la fiche 0006).

    `reference_0003=False` (test du 2e fine-tune) : pas de témoin à 22,1 % (autre test) ; chaque
    modèle est comparé au départ, mesuré dans la même session sur les mêmes clips."""
    base, ft = results[base_label], results[ft_label]
    lines = [f"{'Modele':28s} {'CER':>7s} {'WER':>7s} {'n':>5s} {'vides':>6s}", "-" * 58]
    for label, r in results.items():
        lines.append(f"{label:28s} {r['cer'] * 100:6.1f}% {r['wer'] * 100:6.1f}% {r['n']:5d} {r['empty_hyps']:6d}")
    if not reference_0003:
        lines.append("\nTest du 2e fine-tune (finetune/cv27_split.py) : pas de reference 0003, les modeles sont"
                     " compares au depart dans la meme session.")
        for label, r in results.items():
            if label != base_label:
                lines.append(f"Gain CER {label} : {(base['cer'] - r['cer']) * 100:+.1f} pts "
                             f"({base['cer'] * 100:.1f}% -> {r['cer'] * 100:.1f}%)")
    elif control_ok(base["cer"], base["n"]):
        lines.append(f"\nTEMOIN OK : {base['cer'] * 100:.1f}% (benchmark 0003 : {BENCHMARK_0003_CER * 100:.1f}%)"
                     " -> comparaison recevable.")
    else:
        lines.append(f"\nTEMOIN ECART : {base['cer'] * 100:.1f}% sur {base['n']} clips, attendu "
                     f"{BENCHMARK_0003_CER * 100:.1f}% sur {BENCHMARK_0003_N} -> environnement different, "
                     "a signaler tel quel.")
    lines.append(f"Gain CER : {(base['cer'] - ft['cer']) * 100:+.1f} pts "
                 f"({base['cer'] * 100:.1f}% -> {ft['cer'] * 100:.1f}%)")
    lines.append("\n=== Exemples (reference | depart | fine-tune) ===")
    for i in range(min(len(base["samples"]), len(ft["samples"]))):
        lines.append(f"\n[{i}] REF : {base['samples'][i]['ref']}\n    DEP : {base['samples'][i]['hyp']}"
                     f"\n    FT  : {ft['samples'][i]['hyp']}")
    return "\n".join(lines)


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--checkpoint-root", required=True, help="dossier où le checkpoint a été téléchargé")
    ap.add_argument("--step", default="step_1200", help="étape retenue sur dev (fiche 0006)")
    ap.add_argument("--asset-dir", default=None, help="répertoire de cartes fairseq2 (défaut : temporaire)")
    ap.add_argument("--out", default="eval_bci_cer_results.json")
    ap.add_argument("--limit", type=int, default=None, help="essai rapide sur N clips (défaut : les 290)")
    ap.add_argument("--manifeste", default=None, help="split_cv27.json : test du 2e fine-tune (avec --cv27)")
    ap.add_argument("--cv27", default=None, help="dossier …/bci de Common Voice 27 extrait")
    ap.add_argument("--nouveau", nargs=2, metavar=("RACINE", "ETAPE"), default=None,
                    help="checkpoint du 2e fine-tune, mesuré en plus sur le même test")
    args = ap.parse_args(argv)

    asset_dir = args.asset_dir or tempfile.mkdtemp(prefix="fairseq2_assets_")
    card = write_card(asset_dir, args.step, find_checkpoint(args.checkpoint_root, args.step), "wourri_bci_actuel")
    base_label, ft_label = "temoin 1B depart", f"1B fine-tune {args.step}"
    models = [(base_label, BASE_CARD), (ft_label, card)]
    if args.nouveau:
        racine, etape = args.nouveau
        models.append((f"1B 2e fine-tune {etape}",
                       write_card(asset_dir, etape, find_checkpoint(racine, etape), "wourri_bci_nouveau")))
    # fairseq2 lit ce répertoire à sa première utilisation : à poser avant tout chargement de modèle.
    os.environ["FAIRSEQ2_USER_ASSET_DIR"] = str(asset_dir)

    clips = load_cv27_test(args.manifeste, args.cv27, args.limit) if args.manifeste else None
    results = evaluate(models, args.out, limit=args.limit, clips=clips)
    print("\n" + report(results, base_label, ft_label, reference_0003=clips is None))
    print("\nResultats complets ->", args.out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile serveur_voix_bci.py
"""Serveur voix baoulé (ADR-0045) : le paysan parle, il entend la voix du locuteur natif.

Tourne sur un GPU loué (RunPod), lancé par finetune/colab/serveur_voix_baoule_runpod.ipynb.

- `POST /transcribe` (clé `X-Api-Key`) : le vocal WhatsApp (ogg/opus, multipart `audio`)
  → texte baoulé, par le 1B fine-tuné (fiche 0007 : CER 13,4 %), avec le modèle de langue
  si `BCI_VOIX_LM` désigne le fichier retenu par mesure_lm_bci.py. Le serveur WhatsApp
  envoie ensuite ce texte à `/api/chat/` en langue « baoule ».
- `GET /audio/<nom>.ogg` : les réponses LUES par le locuteur pendant la dictée, converties
  en ogg/opus (format des notes vocales WhatsApp). L'API renvoie leur adresse
  (`BCI_AUDIO_BASE_URL`) ; aucune voix de synthèse.
- `GET /health`.

Réglages (variables d'environnement) : `BCI_VOIX_CLE` (clé partagée avec le serveur
WhatsApp), `BCI_VOIX_CARTE` (carte fairseq2 du modèle, cf. eval_bci_cer.write_card),
`BCI_VOIX_AUDIO` (dossier des réponses ogg), `BCI_VOIX_LM` (facultatif : lm_bci.pkl).

Ce module est recopié tel quel dans le notebook (cellule %%writefile) :
`python finetune/colab/sync_smoke_test_notebook.py` le resynchronise, un test le vérifie.
"""
from __future__ import annotations

import logging
import os
import re
import secrets
import subprocess
import tempfile
import zipfile
from pathlib import Path, PurePosixPath

from fastapi import FastAPI, File, Header, HTTPException, UploadFile
from fastapi.responses import FileResponse

logger = logging.getLogger("voix_bci")

MAX_AUDIO_OCTETS = 10_000_000  # même plafond que la chaîne ASR de l'API
# Le pipeline Omnilingual refuse les audios de plus de 40 s : on garde les 39 premières.
DUREE_MAX_S = 39
NOM_AUDIO = re.compile(r"^[A-Za-z0-9_-]{1,80}\.ogg$")


def vers_wav_16k(data: bytes, dossier: str) -> str:
    """Audio quelconque (ogg/opus de WhatsApp, webm...) → WAV 16 kHz mono, par ffmpeg."""
    chemin = os.path.join(dossier, "vocal.wav")
    proc = subprocess.run(
        ["ffmpeg", "-loglevel", "error", "-y", "-i", "pipe:0", "-t", str(DUREE_MAX_S),
         "-ac", "1", "-ar", "16000", chemin],
        input=data, capture_output=True,
    )
    if proc.returncode != 0:
        raise ValueError(proc.stderr.decode("utf-8", "replace")[-300:] or "ffmpeg a échoué")
    return chemin


def convertir_reponses(export_zip, dossier_ogg) -> int:
    """Chaque audio de l'export Dictée → `<nom>.ogg` (opus copié sans perte, ré-encodé sinon)."""
    out = Path(dossier_ogg)
    out.mkdir(parents=True, exist_ok=True)
    n = 0
    with zipfile.ZipFile(export_zip) as z, tempfile.TemporaryDirectory() as tmp:
        for nom in z.namelist():
            if not nom.lower().endswith((".webm", ".ogg", ".wav", ".m4a", ".mp3")):
                continue
            cible = out / f"{PurePosixPath(nom).stem}.ogg"
            if cible.exists():
                n += 1
                continue
            source = Path(tmp) / PurePosixPath(nom).name
            source.write_bytes(z.read(nom))
            copie = ["ffmpeg", "-loglevel", "error", "-y", "-i", str(source), "-vn", "-c:a", "copy", str(cible)]
            if subprocess.run(copie, capture_output=True).returncode != 0:
                encode = ["ffmpeg", "-loglevel", "error", "-y", "-i", str(source), "-vn",
                          "-c:a", "libopus", "-b:a", "32k", str(cible)]
                subprocess.run(encode, capture_output=True, check=True)
            n += 1
    return n


def _pipeline_par_defaut():
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

    pipe = ASRInferencePipeline(model_card=os.environ["BCI_VOIX_CARTE"])
    if os.environ.get("BCI_VOIX_LM"):
        # Modèle de langue retenu par mesure_lm_bci.py (choisi sur dev) : recherche en faisceau.
        import pickle

        import lm_bci

        with open(os.environ["BCI_VOIX_LM"], "rb") as f:
            cfg = pickle.load(f)
        dec = lm_bci.decodeur(cfg["labels"], lm_bci.modele_pyctcdecode(cfg["trigramme"], cfg["alpha"], cfg["beta"]))
        lm_bci.brancher(pipe, cfg["ids"], cfg["labels"], decodeur=dec)
        logger.info("modele de langue charge (alpha=%s, beta=%s)", cfg["alpha"], cfg["beta"])
    return pipe


def creer_app(pipeline=None, decodeur=vers_wav_16k, cle=None, dossier_audio=None) -> FastAPI:
    """Application FastAPI ; pipeline et décodeur injectables (tests sans GPU ni ffmpeg)."""
    app = FastAPI(title="Wourri — voix baoulé")
    cle = cle if cle is not None else os.environ.get("BCI_VOIX_CLE", "")
    dossier_audio = Path(dossier_audio or os.environ.get("BCI_VOIX_AUDIO", "reponses_ogg"))
    etat = {"pipeline": pipeline}

    def _pipeline():
        if etat["pipeline"] is None:
            etat["pipeline"] = _pipeline_par_defaut()
        return etat["pipeline"]

    @app.get("/health")
    def health():
        return {"ok": True, "modele_charge": etat["pipeline"] is not None,
                "modele_langue": bool(os.environ.get("BCI_VOIX_LM")),
                "reponses_audio": len(list(dossier_audio.glob("*.ogg"))) if dossier_audio.is_dir() else 0}

    @app.post("/transcribe")
    async def transcribe(audio: UploadFile = File(...), x_api_key: str = Header(default="")):
        if not cle or not secrets.compare_digest(x_api_key, cle):
            raise HTTPException(status_code=401, detail="cle invalide")
        data = await audio.read()
        if not data:
            raise HTTPException(status_code=422, detail="audio vide")
        if len(data) > MAX_AUDIO_OCTETS:
            raise HTTPException(status_code=413, detail="audio trop volumineux")
        with tempfile.TemporaryDirectory() as tmp:
            try:
                wav = decodeur(data, tmp)
            except ValueError as exc:
                raise HTTPException(status_code=422, detail=f"audio illisible : {exc}") from exc
            try:
                textes = _pipeline().transcribe([wav], lang=["bci_Latn"], batch_size=1)
            except Exception as exc:  # erreur du modèle : réponse claire, jamais de 500 muet
                logger.exception("transcription impossible")
                raise HTTPException(status_code=422, detail=f"transcription impossible : {type(exc).__name__}") from exc
        return {"text": (list(textes) or [""])[0].strip(), "language": "bci"}

    @app.get("/audio/{nom}")
    def audio_reponse(nom: str):
        if not NOM_AUDIO.match(nom):
            raise HTTPException(status_code=404, detail="introuvable")
        chemin = dossier_audio / nom
        if not chemin.is_file():
            raise HTTPException(status_code=404, detail="introuvable")
        return FileResponse(chemin, media_type="audio/ogg")

    return app


app = creer_app() if os.environ.get("BCI_VOIX_CLE") else None


In [ ]:
%%writefile cv27_split.py
"""Répartition de Common Voice 27 baoulé pour le 2e fine-tune (#474, ADR-0044, révision 2026-10-24).

Le test du benchmark 0003 (290 clips, 9 voix) bloquait 5 h des 11 h validées de Common Voice.
On le remplace par un test choisi par une règle écrite, aussi honnête :
- phrases de test S = les 290 phrases du test officiel : aucune voix ne les lit à l'entraînement ;
- voix de test T = parmi les 9 voix du test officiel, le sous-ensemble qui laisse le plus
  d'heures à l'entraînement en gardant au moins `MIN_CLIPS` clips et `MIN_VOIX` voix au test,
  et `MIN_VOIX_DEV` voix au dev (mesuré le 2026-10-24 : avec une seule voix au dev, 6,77 h
  d'entraînement mais un choix de version fragile ; avec deux, 5,93 h, un dev de 738 clips et
  un test de 623 clips sur 7 voix) ;
- test  = clips validés des voix T sur les phrases S ;
- dev   = clips validés des voix T sur les autres phrases : voix jamais entraînées, texte déjà
  vu. Il sert à CHOISIR la version, jamais à la noter ;
- train = clips validés des autres voix, hors phrases S.

Le modèle actuel (fiche 0006) a été entraîné sur les 3 voix du train officiel et la dictée : il
n'a vu ni les voix T ni les phrases S. Les trois modèles (départ, actuel, nouveau) se comparent
donc sur ce test, dans la même session.

Les phrases sont comparées après `normalize_text` (la normalisation de l'entraînement).
Common Voice interdit le re-hébergement : ce module lit les fichiers sur la machine, il n'écrit
aucun identifiant de voix (le manifeste ne garde que les chemins et les phrases du test).

Ce module est recopié tel quel dans le notebook GPU (cellule %%writefile) :
`python finetune/colab/sync_smoke_test_notebook.py` le resynchronise, un test le vérifie.
"""
from __future__ import annotations

import csv
import itertools
from pathlib import Path

import dictee_to_parquet as d2p

MIN_CLIPS = 290  # autant que le test du benchmark 0003
MIN_VOIX = 4
MIN_VOIX_DEV = 2  # choisir la version sur une seule voix serait fragile


def _tsv(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f, delimiter="\t", quoting=csv.QUOTE_NONE))


def trouver(racine) -> Path:
    """Dossier `…/bci` d'un Common Voice extrait (celui qui contient validated.tsv)."""
    hits = sorted(Path(racine).rglob("validated.tsv"))
    if len(hits) != 1:
        raise SystemExit(f"{len(hits)} validated.tsv sous {racine} : 1 attendu (archive Common Voice extraite)")
    return hits[0].parent


def lire(dossier) -> tuple[list[dict], list[dict]]:
    """(clips validés avec durée en heures et phrase normalisée, clips du test officiel)."""
    d = Path(dossier)
    duree = {r["clip"]: int(r["duration[ms]"]) / 3.6e6 for r in _tsv(d / "clip_durations.tsv")}
    valides = []
    for r in _tsv(d / "validated.tsv"):
        valides.append({"client_id": r["client_id"], "path": r["path"], "sentence": r["sentence"],
                        "cle": d2p.normalize_text(r["sentence"]), "h": duree.get(r["path"], 0.0)})
    return valides, _tsv(d / "test.tsv")


def repartir(valides, phrases_test: set[str], voix_test: set[str]) -> dict[str, list[dict]]:
    rep = {"train": [], "dev": [], "test": []}
    for r in valides:
        if r["client_id"] in voix_test:
            rep["test" if r["cle"] in phrases_test else "dev"].append(r)
        elif r["cle"] not in phrases_test:
            rep["train"].append(r)
    return rep


def choisir_voix_test(valides, phrases_test: set[str], voix_officielles, min_clips=None,
                      min_voix=None, min_voix_dev=None) -> set[str]:
    """Sous-ensemble des voix du test officiel qui maximise les heures d'entraînement.

    À égalité : plus de clips au test, puis ordre des identifiants (résultat reproductible).
    Seuils par défaut : constantes du module, lues à l'appel."""
    min_clips = MIN_CLIPS if min_clips is None else min_clips
    min_voix = MIN_VOIX if min_voix is None else min_voix
    min_voix_dev = MIN_VOIX_DEV if min_voix_dev is None else min_voix_dev
    voix = sorted(voix_officielles)
    meilleur, cle_meilleure = None, None
    for n in range(min_voix, len(voix) + 1):
        for combo in itertools.combinations(voix, n):
            rep = repartir(valides, phrases_test, set(combo))
            if len(rep["test"]) < min_clips or len({r["client_id"] for r in rep["dev"]}) < min_voix_dev:
                continue
            cle = (round(sum(r["h"] for r in rep["train"]), 6), len(rep["test"]))
            if cle_meilleure is None or cle > cle_meilleure:
                meilleur, cle_meilleure = set(combo), cle
    if meilleur is None:
        raise SystemExit(f"aucun choix de voix ne donne {min_clips} clips et {min_voix} voix au test, "
                         f"{min_voix_dev} voix au dev")
    return meilleur


def verifier(rep, phrases_test: set[str], voix_test: set[str], autres_textes=()) -> None:
    """Arrête tout si une voix ou une phrase du test peut être apprise (Common Voice, Zenodo, dictée)."""
    fuites = [r["path"] for r in rep["train"] if r["client_id"] in voix_test or r["cle"] in phrases_test]
    fuites += [t for t in autres_textes if d2p.normalize_text(t) in phrases_test]
    if fuites:
        raise ValueError(f"{len(fuites)} élément(s) du test dans l'entraînement, ex. {fuites[:3]}")
    if not rep["test"] or not rep["dev"] or not rep["train"]:
        raise ValueError("un split est vide : vérifier l'archive Common Voice")


def preparer(dossier, autres_textes=()) -> tuple[dict[str, list[dict]], dict]:
    """Répartition vérifiée + manifeste (sans identifiant de voix)."""
    valides, test_officiel = lire(dossier)
    phrases_test = {d2p.normalize_text(r["sentence"]) for r in test_officiel}
    voix_test = choisir_voix_test(valides, phrases_test, {r["client_id"] for r in test_officiel})
    rep = repartir(valides, phrases_test, voix_test)
    verifier(rep, phrases_test, voix_test, autres_textes)
    manifeste = {
        "regle": "test = voix T x phrases du test officiel ; dev = voix T x autres phrases ; "
                 "train = autres voix hors phrases du test (finetune/cv27_split.py)",
        "splits": {s: {"clips": len(rows), "heures": round(sum(r["h"] for r in rows), 2),
                       "voix": len({r["client_id"] for r in rows})} for s, rows in rep.items()},
        "test": [{"path": r["path"], "sentence": r["sentence"]} for r in rep["test"]],
    }
    return rep, manifeste


In [ ]:
%%writefile lm_bci.py
"""Modèle de langue baoulé pour le décodage CTC d'Omnilingual (#474, ADR-0044, révision du 2026-10-24).

Le CTC du 1B écrit lettre par lettre (décodage « glouton ») et colle ou coupe des mots
(« nansran » pour « naan sran »). Une recherche en faisceau (pyctcdecode) guidée par un modèle
de langue de MOTS corrige une partie de ces erreurs. Le modèle est un trigramme Kneser-Ney
interpolé écrit ici en Python, aux conventions du modèle KenLM de pyctcdecode : testable sans
GPU ni compilation (KenLM demanderait de compiler lmplz sur la machine louée).

Corpus : textes baoulé du registre des sources, normalisés comme les cibles d'entraînement
(`dictee_to_parquet.normalize_text`). Deux règles :
- aucune phrase du test : toute ligne qui en contient une est retirée et comptée (fuite) ;
- pas la Bible (décision du valideur du 2026-10-24 : mot à mot seulement).
α (poids du modèle) et β (bonus par mot) sont choisis sur `dev`, jamais sur `test`.

Vocabulaire du tokenizer omniASR (char_tokenizer) : 0 = blanc CTC (recette : `blank_label=0`),
1–3 = <pad> </s> <unk>, 4 = espace, le reste = un caractère par symbole.

Ce module est recopié tel quel dans les notebooks GPU (cellule %%writefile) :
`python finetune/colab/sync_smoke_test_notebook.py` le resynchronise, un test le vérifie.
"""
from __future__ import annotations

import math
import re
from collections import Counter

import dictee_to_parquet as d2p

BLANC, ESPACE = 0, 4
SPECIAUX = {1, 2, 3}
DISCOUNT = 0.75
UNK_LOG10 = -7.0  # mot inconnu du corpus ; pyctcdecode ajoute en plus son unk_score_offset


def preparer_corpus(textes, phrases_test, min_mots=2) -> tuple[list[str], dict]:
    """Lignes normalisées, sans aucune ligne contenant une phrase du test (avec les comptes)."""
    tests = sorted({d2p.normalize_text(p) for p in phrases_test} - {""}, key=len, reverse=True)
    fuite = re.compile(r"(?<!\S)(?:" + "|".join(re.escape(t) for t in tests) + r")(?!\S)") if tests else None
    lignes, n_fuites = [], 0
    for texte in textes:
        for morceau in re.split(r"[.!?;\n]+", texte or ""):
            ligne = d2p.normalize_text(morceau)
            if len(ligne.split()) < min_mots:
                continue
            if fuite is not None and fuite.search(ligne):
                n_fuites += 1
                continue
            lignes.append(ligne)
    return lignes, {"lignes": len(lignes), "fuites_retirees": n_fuites,
                    "mots": sum(len(l.split()) for l in lignes)}


SEUIL_BAOULE = 0.6


def filtrer_baoule(lignes, reference, seuil=SEUIL_BAOULE) -> tuple[list[str], int]:
    """Garde les lignes dont au moins `seuil` des mots sont dans `reference` (vocabulaire des textes
    natifs fiables : Common Voice, dictée, Zenodo, SMOL). Mesuré le 2026-10-24 sur le corpus web :
    la répartition est bimodale ; sous 50 % ce sont des pages anglaises (annonces, annuaires), du
    français et d'autres langues ivoiriennes étiquetées baoulé ; au-dessus de 60 %, du baoulé."""
    gardees = [l for l in lignes if sum(w in reference for w in l.split()) >= seuil * len(l.split())]
    return gardees, len(lignes) - len(gardees)


class Trigramme:
    """Trigramme Kneser-Ney interpolé (log10, comme KenLM). <s> et </s> bornent chaque ligne."""

    def __init__(self, lignes, min_trigramme=1):
        c3 = Counter()
        for ligne in lignes:
            mots = ["<s>", "<s>"] + ligne.split() + ["</s>"]
            for i in range(2, len(mots)):
                c3[tuple(mots[i - 2:i + 1])] += 1
        self.c3 = {k: v for k, v in c3.items() if v >= min_trigramme}
        self.ctx3, self.types3, self.cont2 = Counter(), Counter(), Counter()
        for (u, v, w), n in self.c3.items():
            self.ctx3[(u, v)] += n          # c(u, v, ·)
            self.types3[(u, v)] += 1        # N1+(u, v, ·)
            self.cont2[(v, w)] += 1         # N1+(·, v, w)
        self.ctx2, self.types2, self.cont1 = Counter(), Counter(), Counter()
        for (v, w), n in self.cont2.items():
            self.ctx2[v] += n               # N1+(·, v, ·)
            self.types2[v] += 1             # N1+(v, ·) parmi les continuations
            self.cont1[w] += 1              # N1+(·, w)
        self.total1 = sum(self.cont1.values())
        self.vocab = set(self.cont1) - {"<s>", "</s>"}

    def p1(self, w) -> float:
        n = self.cont1.get(w, 0)
        return n / self.total1 if n else 0.0

    def p2(self, v, w) -> float:
        base = self.p1(w)
        tot = self.ctx2.get(v, 0)
        if not tot:
            return base
        return max(self.cont2.get((v, w), 0) - DISCOUNT, 0) / tot + DISCOUNT * self.types2[v] / tot * base

    def p3(self, u, v, w) -> float:
        base = self.p2(v, w)
        tot = self.ctx3.get((u, v), 0)
        if not tot:
            return base
        return max(self.c3.get((u, v, w), 0) - DISCOUNT, 0) / tot + DISCOUNT * self.types3[(u, v)] / tot * base

    def log10(self, contexte, w) -> float:
        p = self.p3(contexte[0], contexte[1], w)
        return math.log10(p) if p > 0 else UNK_LOG10


def modele_pyctcdecode(trigramme, alpha, beta, unk_score_offset=-10.0):
    """Modèle de langue pour pyctcdecode : mêmes formules que son modèle KenLM (score_boundary)."""
    from pyctcdecode.language_model import AVG_TOKEN_LEN, LOG_BASE_CHANGE_FACTOR, AbstractLanguageModel, CharTrie

    class ModeleBci(AbstractLanguageModel):
        def __init__(self):
            self.alpha, self.beta, self.unk_score_offset = alpha, beta, unk_score_offset
            self._trie = CharTrie.fromkeys(trigramme.vocab)

        @property
        def order(self) -> int:
            return 3

        def get_start_state(self):
            return ("<s>", "<s>")

        def score_partial_token(self, partial_token: str) -> float:
            unk = self.unk_score_offset * int(self._trie.has_node(partial_token) == 0)
            return unk * len(partial_token) / AVG_TOKEN_LEN if len(partial_token) > AVG_TOKEN_LEN else unk

        def score(self, prev_state, word, is_last_word=False):
            s = trigramme.log10(prev_state, word)
            if word not in trigramme.vocab:
                s += self.unk_score_offset
            etat = (prev_state[1], word)
            if is_last_word:
                s += trigramme.log10(etat, "</s>")
            return self.alpha * s * LOG_BASE_CHANGE_FACTOR + self.beta, etat

    return ModeleBci()


def alphabet_de(textes) -> set[str]:
    """Caractères des textes normalisés (cibles d'entraînement et corpus)."""
    return {c for t in textes for c in d2p.normalize_text(t) if not c.isspace()}


def colonnes(pieces, alphabet) -> tuple[list[int], list[str]]:
    """Symboles gardés pour décoder (blanc d'abord, espace, caractères de l'alphabet baoulé)
    et leurs étiquettes pyctcdecode. Les autres colonnes (milliers de caractères d'autres
    écritures) sont retirées APRÈS le log-softmax : la masse perdue est négligeable."""
    ids = [BLANC, ESPACE] + [i for i, p in enumerate(pieces)
                             if i not in SPECIAUX | {BLANC, ESPACE} and p in alphabet]
    return ids, ["" if i == BLANC else " " if i == ESPACE else pieces[i] for i in ids]


def glouton(matrice, labels) -> str:
    """Décodage CTC glouton sur la matrice réduite (même règle que le pipeline Omnilingual)."""
    out, prec = [], None
    for i in matrice.argmax(axis=1):
        if i != prec and labels[i] != "":
            out.append(labels[i])
        prec = i
    return re.sub(r"\s+", " ", "".join(out)).strip()


def brancher(pipeline, ids, labels, decodeur=None, capture=None, beam=64):
    """Remplace le décodage glouton du pipeline : log-probas réduites aux colonnes `ids`, puis
    recherche en faisceau si `decodeur`, sinon glouton. `capture` (liste) reçoit les matrices."""
    import torch
    from fairseq2.nn import BatchLayout

    index = torch.tensor(ids)

    def appliquer(batch):
        layout = BatchLayout(batch.source_seqs.shape, seq_lens=batch.source_seq_lens, device=batch.source_seqs.device)
        logits, bl_out = pipeline.model(batch.source_seqs, layout)
        lp = torch.log_softmax(logits.float(), dim=-1)[..., index.to(logits.device)].cpu().numpy()
        textes = []
        for i in range(lp.shape[0]):
            m = lp[i, : int(bl_out.seq_lens[i])]
            if capture is not None:
                capture.append(m)
            textes.append(decodeur.decode(m, beam_width=beam) if decodeur else glouton(m, labels))
        return textes

    pipeline._apply_model_wav2vec2asr = appliquer
    return pipeline


def decodeur(labels, modele):
    from pyctcdecode.alphabet import Alphabet
    from pyctcdecode.decoder import BeamSearchDecoderCTC

    return BeamSearchDecoderCTC(Alphabet.build_alphabet(labels), modele)


def decoder_tout(dec, matrices, beam=64, processus=0) -> list[str]:
    """Décode toutes les matrices ; `processus` > 0 : en parallèle (Linux). Le pool est créé APRÈS
    le décodeur : pyctcdecode le retrouve par fork, sans copier le modèle de langue."""
    if processus <= 0:
        return [dec.decode(m, beam_width=beam) for m in matrices]
    import multiprocessing

    with multiprocessing.get_context("fork").Pool(processus) as pool:
        return dec.decode_batch(pool, matrices, beam_width=beam)


def regler(matrices, refs, labels, trigramme, alphas, betas, score, beam=64, processus=0) -> dict:
    """Meilleur (α, β) par CER sur ces clips (dev). `score(refs, hyps)` = protocole 0003."""
    essais = []
    for a in alphas:
        for b in betas:
            hyps = decoder_tout(decodeur(labels, modele_pyctcdecode(trigramme, a, b)), matrices, beam, processus)
            r = score(refs, hyps)
            essais.append({"alpha": a, "beta": b, "cer": r["cer"], "wer": r["wer"]})
    return {"meilleur": min(essais, key=lambda e: (e["cer"], e["wer"])), "essais": essais}


In [ ]:
%%writefile mesure_lm_bci.py
"""Mesure du modèle de langue baoulé sur le 2e fine-tune (#474) — à lancer sur la machine louée.

1. Corpus (lm_bci) : textes natifs (Common Voice validé, dictée, Zenodo, SMOL) et web filtré
   (`filtrer_baoule`), sans aucune phrase de dev NI du test ; pas la Bible. Dev sert à régler α et β :
   s'il était dans le corpus, le modèle de langue l'aurait vu mot pour mot et paraîtrait meilleur
   qu'il n'est (mesuré le 2026-10-24 : couverture des mots de dev 100 % contre 96 % au test).
2. Deux candidats : « natif » (textes natifs seuls) et « complet » (natif + web filtré).
3. Log-probabilités du modèle retenu (fiche 0007) sur dev et test, en une seule passe GPU.
4. Glouton (contrôle : il doit retrouver le CER de la fiche 0007) ; α et β réglés sur dev pour
   chaque candidat ; le candidat est choisi sur dev ; il n'est noté qu'UNE fois sur test.
5. Sorties : `mesure_lm_bci.json` ; `lm_bci.pkl` (modèle et réglages pour le serveur voix), écrit
   seulement si le modèle de langue fait mieux que le glouton sur dev.

Usage (Python 3.12 du notebook) :
    python mesure_lm_bci.py --cv27 /workspace/cv27/.../bci --dictee /workspace/dictee_bci_433.zip \\
        --zenodo /workspace/bci-datasets.zip --checkpoint-root /workspace/ft_nouveau --step step_3000 \\
        --out /workspace/lm --processus 14

Ce module est recopié tel quel dans le notebook (cellule %%writefile) : un test le vérifie.
"""
from __future__ import annotations

import argparse
import csv
import io
import json
import os
import pickle
import tempfile
import urllib.request
import zipfile
from pathlib import Path

import cv27_split as cvs
import dictee_to_parquet as d2p
import lm_bci as lm

HF = "https://huggingface.co/datasets"
WEB = {
    "smolsent_en_bci.jsonl": f"{HF}/google/smol/resolve/main/smolsent/en_bci.jsonl",
    "smoldoc_en_bci.jsonl": f"{HF}/google/smol/resolve/main/smoldoc/en_bci.jsonl",
    "fineweb2_bci.parquet": f"{HF}/HuggingFaceFW/fineweb-2/resolve/main/data/bci_Latn/train/000_00000.parquet",
    "finepdfs_bci.parquet": f"{HF}/HuggingFaceFW/finepdfs/resolve/main/data/bci_Latn/train/000_00000.parquet",
    "glotcc_bci.parquet": f"{HF}/cis-lmu/GlotCC-V1/resolve/main/v1.0/bci-Latn/bci-Latn_0.parquet",
    "madlad_bci.csv": f"{HF}/AfriSpeech/madlad-91-african/resolve/main/Baoul%C3%A9_bci_v2.csv",
}
TOKENIZER = "https://dl.fbaipublicfiles.com/mms/omniASR_tokenizer.model"  # carte omniASR_CTC_1B
ALPHAS = (0.2, 0.4, 0.6, 0.8, 1.0)
BETAS = (0.0, 1.0, 2.0, 3.0)


def telecharger(dossier) -> Path:
    d = Path(dossier)
    d.mkdir(parents=True, exist_ok=True)
    for nom, url in {**WEB, "omniASR_tokenizer.model": TOKENIZER}.items():
        if not (d / nom).exists():
            urllib.request.urlretrieve(url, d / nom)
    return d


def textes_natifs(cv27_dir, dictee, zenodo, sources) -> list[str]:
    cv = [r["sentence"] for r in cvs._tsv(Path(cv27_dir) / "validated_sentences.tsv")]
    dic = [r["transcription"] for r in d2p.read_export(dictee)]
    with zipfile.ZipFile(zenodo) as z:
        zen = [r["bci"] for r in csv.DictReader(io.StringIO(z.read("bci-datasets/transcription.tsv").decode("utf-8-sig")),
                                               delimiter="\t")]
    smol = [t for f in ("smolsent_en_bci.jsonl", "smoldoc_en_bci.jsonl")
            for l in (Path(sources) / f).read_text(encoding="utf-8").splitlines() if l.strip()
            for t in json.loads(l).get("trgs", [])]
    return cv + dic + zen + smol


def textes_web(sources) -> list[str]:
    import pyarrow.parquet as pq

    d = Path(sources)
    docs = []
    for nom, col in (("fineweb2_bci.parquet", "text"), ("finepdfs_bci.parquet", "text"), ("glotcc_bci.parquet", "content")):
        docs += pq.read_table(d / nom, columns=[col]).column(col).to_pylist()
    csv.field_size_limit(10**9)
    with open(d / "madlad_bci.csv", encoding="utf-8") as f:
        docs += [r["local"] for r in csv.DictReader(f)]
    return [str(x) for x in docs if x]


def corpus(natifs, web, phrases_exclues) -> tuple[dict, dict]:
    """Lignes des deux candidats et comptes (phrases de dev et de test retirées, web écarté)."""
    natif, st_n = lm.preparer_corpus(natifs, phrases_exclues)
    ref = {w for l in natif for w in l.split()}
    web_lignes, st_w = lm.preparer_corpus(web, phrases_exclues)
    web_ok, ecartees = lm.filtrer_baoule(web_lignes, ref)
    stats = {"natif": st_n, "web": {**st_w, "ecartees_non_baoule": ecartees, "gardees": len(web_ok)}}
    return {"natif": natif, "complet": natif + web_ok}, stats


def pieces_tokenizer(path) -> list[str]:
    import sentencepiece as spm

    sp = spm.SentencePieceProcessor(model_file=str(path))
    return [sp.id_to_piece(i) for i in range(sp.get_piece_size())]


def capturer(card, rows, cv27_dir, ids, labels) -> tuple[list, list[str]]:
    """(matrices réduites, transcriptions gloutonnes) du modèle, clip par clip, dans l'ordre."""
    import soundfile as sf
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

    pipe = ASRInferencePipeline(model_card=card)
    mats = []
    lm.brancher(pipe, ids, labels, capture=mats)
    tmp = tempfile.mkdtemp()
    paths = []
    for i, r in enumerate(rows):
        wav, _ = d2p.decode_to_16k_mono((Path(cv27_dir) / "clips" / r["path"]).read_bytes(), r["path"])
        p = os.path.join(tmp, f"{i}.wav")
        sf.write(p, wav, d2p.TARGET_SR)
        paths.append(p)
    hyps = list(pipe.transcribe(paths, lang=["bci_Latn"] * len(paths), batch_size=1))
    return mats, hyps


def main(argv=None):
    import eval_bci_cer as ev

    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    for a in ("--cv27", "--dictee", "--zenodo", "--checkpoint-root", "--out"):
        ap.add_argument(a, required=True)
    ap.add_argument("--step", default="step_3000")
    ap.add_argument("--sources", default=None, help="dossier des textes (défaut : <out>/sources)")
    ap.add_argument("--processus", type=int, default=0)
    ap.add_argument("--beam", type=int, default=64)
    args = ap.parse_args(argv)

    out = Path(args.out)
    sources = telecharger(args.sources or out / "sources")
    rep, _ = cvs.preparer(args.cv27)
    phrases_test = [r["sentence"] for r in rep["test"]]
    exclues = phrases_test + [r["sentence"] for r in rep["dev"]]
    candidats, stats = corpus(textes_natifs(args.cv27, args.dictee, args.zenodo, sources), textes_web(sources), exclues)
    print("Corpus :", json.dumps(stats, ensure_ascii=False), flush=True)
    ids, labels = lm.colonnes(pieces_tokenizer(sources / "omniASR_tokenizer.model"), lm.alphabet_de(candidats["natif"]))

    asset_dir = tempfile.mkdtemp(prefix="fairseq2_assets_")
    card = ev.write_card(asset_dir, args.step, ev.find_checkpoint(args.checkpoint_root, args.step), "wourri_bci_nouveau")
    os.environ["FAIRSEQ2_USER_ASSET_DIR"] = asset_dir
    mats, hyps = capturer(card, rep["dev"] + rep["test"], args.cv27, ids, labels)
    n_dev = len(rep["dev"])
    refs = {"dev": [r["sentence"] for r in rep["dev"]], "test": phrases_test}
    m = {"dev": mats[:n_dev], "test": mats[n_dev:]}
    resultats = {"corpus": stats, "colonnes": len(ids),
                 "glouton": {s: ev.score(refs[s], h) for s, h in (("dev", hyps[:n_dev]), ("test", hyps[n_dev:]))}}
    print("Glouton :", {s: round(r["cer"] * 100, 2) for s, r in resultats["glouton"].items()}, flush=True)

    tris = {nom: lm.Trigramme(lignes) for nom, lignes in candidats.items()}
    resultats["candidats"] = {nom: lm.regler(m["dev"], refs["dev"], labels, tri, ALPHAS, BETAS, ev.score,
                                             args.beam, args.processus) for nom, tri in tris.items()}
    nom = min(resultats["candidats"], key=lambda k: resultats["candidats"][k]["meilleur"]["cer"])
    best = resultats["candidats"][nom]["meilleur"]
    dec = lm.decodeur(labels, lm.modele_pyctcdecode(tris[nom], best["alpha"], best["beta"]))
    hyps_test = lm.decoder_tout(dec, m["test"], args.beam, args.processus)
    resultats["retenu"] = {"lm": nom, **best}
    resultats["test"] = {"glouton": resultats["glouton"]["test"], "lm": ev.score(refs["test"], hyps_test)}
    out.mkdir(parents=True, exist_ok=True)
    (out / "mesure_lm_bci.json").write_text(json.dumps(resultats, ensure_ascii=False, indent=1, default=str),
                                            encoding="utf-8")
    if best["cer"] < resultats["glouton"]["dev"]["cer"]:
        with open(out / "lm_bci.pkl", "wb") as f:
            pickle.dump({"trigramme": tris[nom], "alpha": best["alpha"], "beta": best["beta"],
                         "ids": ids, "labels": labels}, f)
    g, l = resultats["test"]["glouton"], resultats["test"]["lm"]
    print(f"\nRetenu sur dev : modèle « {nom} », α={best['alpha']}, β={best['beta']} "
          f"(CER dev {resultats['glouton']['dev']['cer'] * 100:.1f} % -> {best['cer'] * 100:.1f} %)")
    print(f"TEST ({len(phrases_test)} clips) : CER {g['cer'] * 100:.1f} % -> {l['cer'] * 100:.1f} % | "
          f"WER {g['wer'] * 100:.1f} % -> {l['wer'] * 100:.1f} %")
    print("Serveur voix :", (out / "lm_bci.pkl") if (out / "lm_bci.pkl").exists()
          else "glouton (le modèle de langue ne gagne pas sur dev)")


if __name__ == "__main__":
    main()


## Étape 3 — Le modèle fine-tuné (dépôt privé, ~3,9 Go)

Une **case de saisie masquée** apparaît : colle le jeton Hugging Face (lecture seule), puis Entrée.

In [ ]:
# Cellule 7 — télécharger le modèle et écrire sa carte fairseq2
from huggingface_hub import snapshot_download
import getpass

_token = getpass.getpass("Jeton Hugging Face (lecture seule) : ")
try:
    snapshot_download(HF_REPO, allow_patterns=[f"{STEP}/*"], local_dir=str(CKPT_ROOT), token=_token)
finally:
    del _token
code = (
    "import eval_bci_cer as ev; "
    f"print(ev.write_card(r'{ASSETS}', '{STEP}', ev.find_checkpoint(r'{CKPT_ROOT}', '{STEP}')))"
)
r = subprocess.run([VENV_PY, "-c", code], capture_output=True, text=True, env=clean_env(), cwd=str(WORK))
if r.returncode != 0:
    raise SystemExit(r.stderr[-2000:])
CARTE = r.stdout.strip().splitlines()[-1]
print("Carte du modèle :", CARTE)
print(f"\n>>> MODÈLE {STEP} PRÊT -> cellule 8.")

## Étape 3 bis (facultative) — Le modèle de langue

Mesure le décodage **avec** un modèle de langue baoulé (textes du registre, sans aucune phrase du test, sans la Bible) contre le décodage simple. Réglages choisis sur dev ; le serveur ne l'utilise **que s'il gagne sur dev**. Une saisie masquée demande la clé API Mozilla.

In [ ]:
# Cellule 7b (FACULTATIVE, ~25 min) — modèle de langue : choisi sur dev, noté une fois sur test
import hashlib, json, urllib.error, urllib.request

UA = {"User-Agent": "wourri-finetune/1.0 (+https://github.com/ouedraogoissouf2012/wourri)"}
CV27_ID = "cmu625dl700nxnq07jev059gj"          # Common Voice Scripted Speech 27.0 - Baoule
CV27_TAR = WORK / "cv27_bci.tar.gz"
if not CV27_TAR.exists():
    _cle = getpass.getpass("Clé API Mozilla Data Collective : ").strip()
    _req = urllib.request.Request(
        f"https://mozilladatacollective.com/api/datasets/{CV27_ID}/download", data=b"{}", method="POST",
        headers={**UA, "Authorization": f"Bearer {_cle}", "Content-Type": "application/json", "Accept": "application/json"})
    del _cle
    try:
        _info = json.load(urllib.request.urlopen(_req, timeout=60))
    except urllib.error.HTTPError as e:
        raise SystemExit(f"Mozilla refuse : {e.code} {e.read()[:300].decode('utf-8', 'replace')}")
    with urllib.request.urlopen(urllib.request.Request(_info["downloadUrl"], headers=UA), timeout=900) as r, \
         open(CV27_TAR, "wb") as f:
        shutil.copyfileobj(r, f)
ZENODO_ZIP = WORK / "bci-datasets.zip"
if not ZENODO_ZIP.exists():
    with urllib.request.urlopen(urllib.request.Request(
            "https://zenodo.org/records/6705861/files/bci-datasets.zip?download=1", headers=UA), timeout=900) as r, \
         open(ZENODO_ZIP, "wb") as f:
        shutil.copyfileobj(r, f)
if hashlib.md5(ZENODO_ZIP.read_bytes()).hexdigest() != "0af939f1f4672ec887792ba1715ae3c1":
    ZENODO_ZIP.unlink()
    raise SystemExit("Zenodo : somme de contrôle différente, relancer la cellule")
r = subprocess.run([VENV_PY, "-c", f"import prepare_bci_dataset as p; print(p.extraire_cv27(r'{CV27_TAR}', r'{WORK}'))"],
                   capture_output=True, text=True, env=clean_env(), cwd=str(WORK))
if r.returncode != 0:
    raise SystemExit(r.stderr[-2000:])
CV27_DIR = r.stdout.strip().splitlines()[-1]
sh([VENV_PY, "mesure_lm_bci.py", "--cv27", CV27_DIR, "--dictee", EXPORT_ZIP, "--zenodo", ZENODO_ZIP,
    "--checkpoint-root", CKPT_ROOT, "--step", STEP, "--out", LM_DIR, "--processus", str(max(1, (os.cpu_count() or 2) - 2))],
   cwd=str(WORK), env=clean_env({"CUDA_VISIBLE_DEVICES": "0"}))
print("\n>>> MESURE TERMINÉE. Envoie les lignes « Corpus », « Glouton », « Retenu » et « TEST ».")


## Étape 4 — Les réponses lues par le locuteur

Téléverse d'abord `dictee_bci_433.zip` dans `/workspace` (flèche vers le haut du panneau de fichiers).

In [ ]:
# Cellule 8 — convertir les enregistrements de la dictée en notes vocales WhatsApp (ogg/opus)
if not EXPORT_ZIP.is_file():
    raise SystemExit(f"Téléverse d'abord {EXPORT_ZIP.name} dans {WORK}")
sh([VENV_PY, "-c", f"import serveur_voix_bci as v; print('Réponses converties :', v.convertir_reponses(r'{EXPORT_ZIP}', r'{AUDIO_DIR}'))"],
   cwd=str(WORK))
print(f"\n>>> {len(list(AUDIO_DIR.glob('*.ogg')))} ENREGISTREMENTS PRÊTS -> cellule 9.")

## Étape 5 — Lancer le serveur

La clé (partagée avec le serveur WhatsApp) est créée une fois et gardée dans `/workspace/voix_cle.txt`.
**C'est prêt quand les réglages à copier s'affichent.**

In [ ]:
# Cellule 9 — lancer le serveur voix (port 8000) et le préchauffer
import secrets, time, urllib.request

CLE_FILE = WORK / "voix_cle.txt"
if not CLE_FILE.exists():
    CLE_FILE.write_text(secrets.token_urlsafe(24))
CLE = CLE_FILE.read_text().strip()
env = clean_env({"BCI_VOIX_CLE": CLE, "BCI_VOIX_CARTE": CARTE, "BCI_VOIX_AUDIO": str(AUDIO_DIR),
                 "FAIRSEQ2_USER_ASSET_DIR": str(ASSETS), "CUDA_VISIBLE_DEVICES": "0"})
LM_PKL = LM_DIR / "lm_bci.pkl"
if LM_PKL.exists():
    env["BCI_VOIX_LM"] = str(LM_PKL)              # retenu sur dev par la cellule 7b
print("Décodage :", "avec modèle de langue" if LM_PKL.exists() else "simple (glouton)")
subprocess.run(["pkill", "-f", "uvicorn serveur_voix_bci"], check=False)
log = open(WORK / "voix.log", "w")
SERVEUR = subprocess.Popen([VENV_PY, "-m", "uvicorn", "serveur_voix_bci:app", "--host", "0.0.0.0", "--port", str(PORT)],
                           cwd=str(WORK), env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("Santé :", urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2).read().decode())
        break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit("Le serveur ne démarre pas : voir /workspace/voix.log")

# Préchauffage : une vraie réponse lue par le locuteur, transcrite une fois (charge le modèle)
exemple = sorted(AUDIO_DIR.glob("*.ogg"))[0]
code = (
    "import requests; "
    f"r = requests.post('http://127.0.0.1:{PORT}/transcribe', headers={{'X-Api-Key': '{CLE}'}}, "
    f"files={{'audio': open(r'{exemple}', 'rb')}}, timeout=300); print('Préchauffage :', r.status_code, r.text[:300])"
)
sh([VENV_PY, "-c", code])

URL = f"https://{os.environ.get('RUNPOD_POD_ID', '<pod>')}-{PORT}.proxy.runpod.net"
print("\n================ RÉGLAGES À COPIER ================")
print("API Wourri       : BCI_AUDIO_BASE_URL   =", URL + "/audio")
print("Serveur WhatsApp : WOURI_BAOULE_ASR_URL =", URL)
print("Serveur WhatsApp : WOURI_BAOULE_ASR_KEY =", CLE)
print("Serveur WhatsApp : WOURI_BAOULE_SERVI   = 1   (seulement avec le dictionnaire baoulé VALIDÉ déployé)")
print("===================================================")
print(">>> SERVEUR PRÊT -> cellule 10 (vérification depuis Internet).")

In [ ]:
# Cellule 10 — vérifier l'adresse publique (celle qu'utiliseront l'API et le serveur WhatsApp)
try:
    print("Santé (public) :", urllib.request.urlopen(URL + "/health", timeout=20).read().decode())
    print(">>> L'ADRESSE PUBLIQUE RÉPOND.")
except Exception as exc:
    print("ÉCHEC :", exc)
    print("-> Le port 8000 est-il exposé ? RunPod : Edit Pod -> Expose HTTP Ports -> 8000.")

## Fin de la démo

RunPod : **Stop** puis **Terminate** le pod (sinon il reste facturé). Le modèle reste sur Hugging Face ; tout se relance avec ce notebook.